In [ ]:
import os
import joblib
import tarfile
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# 1. Train a sample Random Forest Regressor
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 2. Save the trained model artifact
os.makedirs("export/code", exist_ok=True)
joblib.dump(model, "export/model.joblib")

# 3. Create SageMaker's required entry-point script
inference_code = """import os
import joblib
import json
import numpy as np

def model_fn(model_dir):
    \"\"\"Loads the model from the local directory inside the container.\"\"\"
    model_path = os.path.join(model_dir, 'model.joblib')
    return joblib.load(model_path)

def input_fn(request_body, request_content_type):
    \"\"\"Parses JSON input array into a numpy array.\"\"\"
    if request_content_type == 'application/json':
        data = json.loads(request_body)
        return np.array(data['inputs'])
    raise ValueError(f"Unsupported content type: {request_content_type}")

def predict_fn(input_data, model):
    \"\"\"Runs inference on the parsed input array.\"\"\"
    return model.predict(input_data)

def output_fn(prediction, accept):
    \"\"\"Formats predictions back to JSON.\"\"\"
    if accept == 'application/json':
        return json.dumps({"predictions": prediction.tolist()}), accept
    raise ValueError(f"Unsupported accept type: {accept}")
"""

with open("export/code/inference.py", "w") as f:
    f.write(inference_code)

# 4. Package as model.tar.gz
with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("export/model.joblib", arcname="model.joblib")
    tar.add("export/code/inference.py", arcname="code/inference.py")

print("Created model.tar.gz ready for S3 upload.")

Created model.tar.gz ready for S3 upload.


In [ ]:
!pip install boto3

import os
import tarfile
import boto3

# The model and inference script were packaged into 'model.tar.gz' by the previous step.
# We will use that existing 'model.tar.gz' for uploading.
print("Using existing model.tar.gz from previous step.")

# Configure AWS credentials
# IMPORTANT: Replace 'YOUR_AWS_ACCESS_KEY_ID' and 'YOUR_AWS_SECRET_ACCESS_KEY' with your actual AWS credentials.
# Do not hardcode sensitive information in production environments.
# Replace real keys like AKIA... with this:
aws_access_key_id = "YOUR_AWS_ACCESS_KEY_ID"
aws_secret_access_key = "YOUR_AWS_SECRET_ACCESS_KEY"

# 3. Upload to S3
s3 = boto3.client("s3", region_name="ap-south-1")
s3.upload_file("model.tar.gz", "rf-housing-model1", "model.tar.gz")
print("Uploaded updated model.tar.gz to s3://rf-housing-model1/model.tar.gz")

Using existing model.tar.gz from previous step.
Uploaded updated model.tar.gz to s3://rf-housing-model1/model.tar.gz


In [ ]:
import requests

url = "https://on6l8u1qcb.execute-api.ap-south-1.amazonaws.com/prod"

payload = {
    "inputs": [
        [8.3252, 41.0, 6.9841, 1.0238, 322.0, 2.5555, 37.88, -122.23]
    ]
}

response = requests.post(url, json=payload)
print("Status Code:", response.status_code)
print("Response:", response.json())

Status Code: 500
Response: {'error': 'An error occurred (ValidationError) when calling the InvokeEndpoint operation: Endpoint rf-housing-endpoint of account 767261813063 not found.'}


In [ ]:
import boto3
import time

sagemaker = boto3.client("sagemaker", region_name="ap-south-1")

endpoint_name = "rf-housing-endpoint"
config_name = "rf-housing-serverless-cfg-v4"
model_name = "rf-housing-model"

# 1. Clean up any existing config with this name
try:
    sagemaker.delete_endpoint_config(EndpointConfigName=config_name)
except Exception:
    pass

# 2. Create Serverless Endpoint Configuration
print("Creating Serverless Endpoint Configuration...")
sagemaker.create_endpoint_config(
    EndpointConfigName=config_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "ServerlessConfig": {
                "MemorySizeInMB": 2048,  # Giving 2GB RAM to ensure smooth loading
                "MaxConcurrency": 2
            }
        }
    ]
)
print("Serverless config created.")

# 3. Create the Endpoint
print("Creating Serverless Endpoint...")
sagemaker.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=config_name
)
print("Endpoint deployment initiated. Waiting for InService status...")

# 4. Poll until InService
while True:
    desc = sagemaker.describe_endpoint(EndpointName=endpoint_name)
    status = desc["EndpointStatus"]
    print("Current Status:", status)
    if status == "InService":
        print(" Endpoint is now InService and ready to use!")
        break
    elif status == "Failed":
        print(" Failed reason:", desc.get("FailureReason"))
        break
    time.sleep(20)

Creating Serverless Endpoint Configuration...
Serverless config created.
Creating Serverless Endpoint...
Endpoint deployment initiated. Waiting for InService status...
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Failed
 Failed reason: Unable to successfully stand up your model within the allotted 180 second timeout. Please ensure that downloading your model artifacts, starting your model container and passing the ping health checks can be completed within 180 seconds.


In [ ]:
import os
import tarfile
import boto3
import time

s3 = boto3.client("s3", region_name="ap-south-1")
sagemaker = boto3.client("sagemaker", region_name="ap-south-1")

bucket_name = "rf-housing-model1"
role_arn = "arn:aws:iam::767261813063:role/SageMakerExecutionRoleForHousePrice"
image_uri = "720646828776.dkr.ecr.ap-south-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3"

model_name = "rf-housing-model-v2"
config_name = "rf-serverless-config-v5"
endpoint_name = "rf-housing-endpoint"

# 1. Clean up old failed endpoint & config
print("Cleaning up previous failed endpoint...")
try:
    sagemaker.delete_endpoint(EndpointName=endpoint_name)
    time.sleep(5)
except Exception:
    pass

try:
    sagemaker.delete_endpoint_config(EndpointConfigName=config_name)
except Exception:
    pass

# 2. Re-create Model with explicit environment variables
print(f"Creating Model: {model_name}...")
sagemaker.create_model(
    ModelName=model_name,
    PrimaryContainer={
        "Image": image_uri,
        "ModelDataUrl": f"s3://{bucket_name}/model.tar.gz",
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": f"s3://{bucket_name}/model.tar.gz"
        }
    },
    ExecutionRoleArn=role_arn
)
print("Model registered successfully.")

# 3. Create Serverless Endpoint Configuration
print("Creating Serverless Endpoint Configuration...")
sagemaker.create_endpoint_config(
    EndpointConfigName=config_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "ServerlessConfig": {
                "MemorySizeInMB": 2048,
                "MaxConcurrency": 2
            }
        }
    ]
)
print("Serverless config created.")

# 4. Create Endpoint
print("Deploying Endpoint...")
sagemaker.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=config_name
)

# 5. Monitor Status
while True:
    desc = sagemaker.describe_endpoint(EndpointName=endpoint_name)
    status = desc["EndpointStatus"]
    print("Current Status:", status)
    if status == "InService":
        print("\n Endpoint is now InService and ready for predictions!")
        break
    elif status == "Failed":
        print("\n Failed reason:", desc.get("FailureReason"))
        break
    time.sleep(15)

Cleaning up previous failed endpoint...
Creating Model: rf-housing-model-v2...
Model registered successfully.
Creating Serverless Endpoint Configuration...
Serverless config created.
Deploying Endpoint...
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Failed

 Failed reason: Received server error (0) from model with message "An error occurred while handling request as the model process exited.". See https://ap-south-1.console.aws.amazon.com/cloudwatch/home?region=ap-south-1#logEventViewer:group=/aws/sagemaker/Endpoints/rf-housing-endpoint in account 767261813063 for more information.


In [ ]:
import os
import tarfile
import boto3
import time

s3 = boto3.client("s3", region_name="ap-south-1")
sagemaker = boto3.client("sagemaker", region_name="ap-south-1")

bucket_name = "rf-housing-model1"
role_arn = "arn:aws:iam::767261813063:role/SageMakerExecutionRoleForHousePrice"
image_uri = "720646828776.dkr.ecr.ap-south-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3"

model_name = "rf-housing-model-v3"
config_name = "rf-serverless-config-v6"
endpoint_name = "rf-housing-endpoint"

# 1. Create a bulletproof inference.py
os.makedirs("code", exist_ok=True)
with open("code/inference.py", "w") as f:
    f.write('''import os
import joblib
import json
import numpy as np

def model_fn(model_dir):
    try:
        # The model is saved as model.joblib in the root of the tarball
        model_path = os.path.join(model_dir, "model.joblib")
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"Model file not found at {model_path}. Available: {os.listdir(model_dir)}")

        model = joblib.load(model_path)
        print("Model successfully loaded.")
        return model
    except Exception as e:
        print(f"CRITICAL: Failed to load model: {e}")
        raise e

def input_fn(request_body, request_content_type):
    if request_content_type == "application/json":
        data = json.loads(request_body)
        if isinstance(data, dict) and "inputs" in data:
            return np.array(data["inputs"])
        return np.array(data)
    raise ValueError(f"Unsupported content type: {request_content_type}")

def predict_fn(input_data, model):
    return model.predict(input_data)

def output_fn(prediction, accept):
    if accept == "application/json":
        return json.dumps(prediction.tolist()), accept
    raise ValueError(f"Unsupported accept type: {accept}")
''')

# 2. Package model.tar.gz cleanly
print("Repackaging model.tar.gz...")
with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("export/model.joblib", arcname="model.joblib") # Corrected to use existing model artifact
    tar.add("code/inference.py", arcname="code/inference.py")

# 3. Upload to S3
print("Uploading model.tar.gz to S3...")
s3.upload_file("model.tar.gz", bucket_name, "model.tar.gz")

# 4. Cleanup old failed endpoint & config
print("Cleaning up old endpoint resources...")
try:
    sagemaker.delete_endpoint(EndpointName=endpoint_name)
    time.sleep(5)
except Exception:
    pass

try:
    sagemaker.delete_endpoint_config(EndpointConfigName=config_name)
except Exception:
    pass

# 5. Create Model
print(f"Creating Model: {model_name}...")
sagemaker.create_model(
    ModelName=model_name,
    PrimaryContainer={
        "Image": image_uri,
        "ModelDataUrl": f"s3://{bucket_name}/model.tar.gz",
        "Environment": {
            "SAGEMAKER_PROGRAM": "code/inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": f"s3://{bucket_name}/model.tar.gz" # Point to the S3 URI of the tarball
        }
    },
    ExecutionRoleArn=role_arn
)

# 6. Create Endpoint Config
print("Creating Serverless Endpoint Configuration...")
sagemaker.create_endpoint_config(
    EndpointConfigName=config_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "ServerlessConfig": {
                "MemorySizeInMB": 2048,
                "MaxConcurrency": 2
            }
        }
    ]
)

# 7. Deploy Endpoint
print("Deploying Endpoint...")
sagemaker.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=config_name
)

# 8. Monitor Status
while True:
    desc = sagemaker.describe_endpoint(EndpointName=endpoint_name)
    status = desc["EndpointStatus"]
    print("Current Status:", status)
    if status == "InService":
        print("\n Endpoint is now InService and ready for predictions!")
        break
    elif status == "Failed":
        print("\n Failed reason:", desc.get("FailureReason"))
        break
    time.sleep(15)

Repackaging model.tar.gz...
Uploading model.tar.gz to S3...
Cleaning up old endpoint resources...
Creating Model: rf-housing-model-v3...
Creating Serverless Endpoint Configuration...
Deploying Endpoint...
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Failed

 Failed reason: Received server error (0) from model with message "An error occurred while handling request as the model process exited.". See https://ap-south-1.console.aws.amazon.com/cloudwatch/home?region=ap-south-1#logEventViewer:group=/aws/sagemaker/Endpoints/rf-housing-endpoint in account 767261813063 for more information.


In [ ]:
import os
import tarfile
import boto3
import time

s3 = boto3.client("s3", region_name="ap-south-1")
sagemaker = boto3.client("sagemaker", region_name="ap-south-1")

bucket_name = "rf-housing-model1"
role_arn = "arn:aws:iam::767261813063:role/SageMakerExecutionRoleForHousePrice"
image_uri = "720646828776.dkr.ecr.ap-south-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3"

model_name = "rf-housing-model-v7"
config_name = "rf-serverless-config-v10"
endpoint_name = "rf-housing-endpoint"

# 1. Package a clean model.tar.gz containing model.joblib at the root
print("Repackaging clean model.tar.gz...")
with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("model.joblib", arcname="model.joblib")

# 2. Upload to S3
print("Uploading clean model.tar.gz to S3...")
s3.upload_file("model.tar.gz", bucket_name, "model.tar.gz")

# 3. Clean up old resources
print("Cleaning up old resources...")
try:
    sagemaker.delete_endpoint(EndpointName=endpoint_name)
    time.sleep(5)
except Exception:
    pass

try:
    sagemaker.delete_endpoint_config(EndpointConfigName=config_name)
except Exception:
    pass

# 4. Register Model without conflicting environment variables
print(f"Registering Model: {model_name}...")
sagemaker.create_model(
    ModelName=model_name,
    PrimaryContainer={
        "Image": image_uri,
        "ModelDataUrl": f"s3://{bucket_name}/model.tar.gz"
    },
    ExecutionRoleArn=role_arn
)

# 5. Create Serverless Endpoint Configuration
print("Creating Serverless Endpoint Configuration...")
sagemaker.create_endpoint_config(
    EndpointConfigName=config_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "ServerlessConfig": {
                "MemorySizeInMB": 2048,
                "MaxConcurrency": 2
            }
        }
    ]
)

# 6. Deploy Endpoint
print("Deploying Serverless Endpoint...")
sagemaker.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=config_name
)

# 7. Monitor Status
while True:
    desc = sagemaker.describe_endpoint(EndpointName=endpoint_name)
    status = desc["EndpointStatus"]
    print("Current Status:", status)
    if status == "InService":
        print("\nEndpoint is now InService and ready for inference!")
        break
    elif status == "Failed":
        print("\nFailed reason:", desc.get("FailureReason"))
        break
    time.sleep(15)

Repackaging clean model.tar.gz...
Uploading clean model.tar.gz to S3...
Cleaning up old resources...
Registering Model: rf-housing-model-v7...
Creating Serverless Endpoint Configuration...
Deploying Serverless Endpoint...
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Failed

Failed reason: Unable to successfully stand up your model within the allotted 180 second timeout. Please ensure that downloading your model artifacts, starting your model container and passing the ping health checks can be comple

In [ ]:
import sklearn
import joblib

print("Current Colab Scikit-Learn version:", sklearn.__version__)

try:
    model = joblib.load("model.joblib")
    print("Model loaded successfully in Colab.")
except Exception as e:
    print("Error loading model.joblib in Colab:", e)

Current Colab Scikit-Learn version: 1.6.1
Model loaded successfully in Colab.


In [ ]:
!pip install scikit-learn==1.2.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 41.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [ ]:
import os
import tarfile
import boto3
import time

s3 = boto3.client("s3", region_name="ap-south-1")
sagemaker = boto3.client("sagemaker", region_name="ap-south-1")

bucket_name = "rf-housing-model1"
role_arn = "arn:aws:iam::767261813063:role/SageMakerExecutionRoleForHousePrice"
# Official SageMaker Scikit-learn 1.4-2 image in ap-south-1
image_uri = "720646828776.dkr.ecr.ap-south-1.amazonaws.com/sagemaker-scikit-learn:1.4-2-cpu-py3"

model_name = "rf-housing-model-v9"
config_name = "rf-serverless-config-v12"
endpoint_name = "rf-housing-endpoint"

# 1. Prepare code/ directory with requirements.txt and inference.py
os.makedirs("code", exist_ok=True)

with open("code/requirements.txt", "w") as f:
    f.write("scikit-learn==1.6.1\njoblib\nnumpy\n")

inference_code = '''import os
import joblib
import json
import numpy as np

def model_fn(model_dir):
    for fname in ["model.joblib", "house_price_model.pkl"]:
        path = os.path.join(model_dir, fname)
        if os.path.exists(path):
            return joblib.load(path)
    candidates = [f for f in os.listdir(model_dir) if f.endswith((".joblib", ".pkl"))]
    if candidates:
        return joblib.load(os.path.join(model_dir, candidates[0]))
    raise FileNotFoundError(f"No model found in {model_dir}: {os.listdir(model_dir)}")

def input_fn(request_body, request_content_type):
    if request_content_type == "application/json":
        data = json.loads(request_body)
        if isinstance(data, dict) and "inputs" in data:
            return np.array(data["inputs"])
        return np.array(data)
    raise ValueError(f"Unsupported content type: {request_content_type}")

def predict_fn(input_data, model):
    return model.predict(input_data)

def output_fn(prediction, accept):
    if accept == "application/json":
        return json.dumps(prediction.tolist()), accept
    raise ValueError(f"Unsupported accept type: {accept}")
'''

with open("code/inference.py", "w") as f:
    f.write(inference_code)

# 2. Package model.joblib together with code/ directory
print("Packaging model.tar.gz...")
with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("model.joblib", arcname="model.joblib")
    tar.add("code/inference.py", arcname="code/inference.py")
    tar.add("code/requirements.txt", arcname="code/requirements.txt")

# 3. Upload to S3
print("Uploading updated model.tar.gz to S3...")
s3.upload_file("model.tar.gz", bucket_name, "model.tar.gz")

# 4. Clean up old resources
print("Cleaning up stale resources...")
try:
    sagemaker.delete_endpoint(EndpointName=endpoint_name)
    time.sleep(5)
except Exception:
    pass

try:
    sagemaker.delete_endpoint_config(EndpointConfigName=config_name)
except Exception:
    pass

# 5. Create Model pointing to the updated 1.4-2 container
print(f"Creating Model: {model_name}...")
sagemaker.create_model(
    ModelName=model_name,
    PrimaryContainer={
        "Image": image_uri,
        "ModelDataUrl": f"s3://{bucket_name}/model.tar.gz",
        "Environment": {
            "SAGEMAKER_PROGRAM": "code/inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": f"s3://{bucket_name}/model.tar.gz"
        }
    },
    ExecutionRoleArn=role_arn
)

# 6. Create Serverless Endpoint Configuration
print("Creating Serverless Endpoint Configuration...")
sagemaker.create_endpoint_config(
    EndpointConfigName=config_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "ServerlessConfig": {
                "MemorySizeInMB": 2048,
                "MaxConcurrency": 2
            }
        }
    ]
)

# 7. Deploy Endpoint
print("Deploying Serverless Endpoint...")
sagemaker.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=config_name
)

# 8. Monitor Status
while True:
    desc = sagemaker.describe_endpoint(EndpointName=endpoint_name)
    status = desc["EndpointStatus"]
    print("Current Status:", status)
    if status == "InService":
        print("\nEndpoint is now InService and ready for predictions!")
        break
    elif status == "Failed":
        print("\nFailed reason:", desc.get("FailureReason"))
        break
    time.sleep(15)

Packaging model.tar.gz...
Uploading updated model.tar.gz to S3...
Cleaning up stale resources...
Creating Model: rf-housing-model-v9...
Creating Serverless Endpoint Configuration...
Deploying Serverless Endpoint...
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current St

In [ ]:
import os
import tarfile
import boto3
import time

s3 = boto3.client("s3", region_name="ap-south-1")
sagemaker = boto3.client("sagemaker", region_name="ap-south-1")

bucket_name = "rf-housing-model1"
role_arn = "arn:aws:iam::767261813063:role/SageMakerExecutionRoleForHousePrice"
image_uri = "720646828776.dkr.ecr.ap-south-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3"

model_name = "rf-housing-model-v10"
config_name = "rf-serverless-config-v13"
endpoint_name = "rf-housing-endpoint"

# 1. Write clean inference script
os.makedirs("code", exist_ok=True)
inference_code = '''import os
import joblib
import json
import numpy as np

def model_fn(model_dir):
    for name in ["model.joblib", "house_price_model.pkl"]:
        path = os.path.join(model_dir, name)
        if os.path.exists(path):
            return joblib.load(path)
    # Fallback to any joblib/pkl found
    for f in os.listdir(model_dir):
        if f.endswith((".joblib", ".pkl")):
            return joblib.load(os.path.join(model_dir, f))
    raise FileNotFoundError(f"No model found in {model_dir}")

def input_fn(request_body, request_content_type):
    if request_content_type == "application/json":
        data = json.loads(request_body)
        if isinstance(data, dict) and "inputs" in data:
            return np.array(data["inputs"])
        return np.array(data)
    raise ValueError(f"Unsupported content type: {request_content_type}")

def predict_fn(input_data, model):
    return model.predict(input_data)

def output_fn(prediction, accept):
    if accept == "application/json":
        return json.dumps(prediction.tolist()), accept
    raise ValueError(f"Unsupported accept type: {accept}")
'''

with open("code/inference.py", "w") as f:
    f.write(inference_code)

with open("inference.py", "w") as f:
    f.write(inference_code)

# 2. Package tarball with root and code/ copies
print("Building model.tar.gz...")
with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("model.joblib", arcname="model.joblib")
    tar.add("inference.py", arcname="inference.py")
    tar.add("code/inference.py", arcname="code/inference.py")

# 3. Upload to S3
print("Uploading model.tar.gz to S3...")
s3.upload_file("model.tar.gz", bucket_name, "model.tar.gz")

# 4. Clean up stale endpoint/config
print("Cleaning up old resources...")
try:
    sagemaker.delete_endpoint(EndpointName=endpoint_name)
    time.sleep(5)
except Exception:
    pass

try:
    sagemaker.delete_endpoint_config(EndpointConfigName=config_name)
except Exception:
    pass

# 5. Create Model with bare module name and expanded PYTHONPATH
print(f"Creating Model: {model_name}...")
sagemaker.create_model(
    ModelName=model_name,
    PrimaryContainer={
        "Image": image_uri,
        "ModelDataUrl": f"s3://{bucket_name}/model.tar.gz",
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference",
            "PYTHONPATH": "/opt/ml/model:/opt/ml/model/code:$PYTHONPATH"
        }
    },
    ExecutionRoleArn=role_arn
)

# 6. Create Serverless Endpoint Config
print("Creating Serverless Endpoint Configuration...")
sagemaker.create_endpoint_config(
    EndpointConfigName=config_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "ServerlessConfig": {
                "MemorySizeInMB": 2048,
                "MaxConcurrency": 2
            }
        }
    ]
)

# 7. Deploy Endpoint
print("Deploying Serverless Endpoint...")
sagemaker.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=config_name
)

# 8. Monitor Status
while True:
    desc = sagemaker.describe_endpoint(EndpointName=endpoint_name)
    status = desc["EndpointStatus"]
    print("Current Status:", status)
    if status == "InService":
        print("\nEndpoint is now InService and ready for predictions!")
        break
    elif status == "Failed":
        print("\nFailed reason:", desc.get("FailureReason"))
        break
    time.sleep(15)

Building model.tar.gz...
Uploading model.tar.gz to S3...
Cleaning up old resources...
Creating Model: rf-housing-model-v10...
Creating Serverless Endpoint Configuration...
Deploying Serverless Endpoint...
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Failed

Failed reason: Unable to successfully stand up your model within the allotted 180 second timeout. Please ensure that downloading your model artifacts, starting your model container and passing the ping health checks can be completed within 180 se

In [ ]:
import os
import tarfile
import boto3
import time

s3 = boto3.client("s3", region_name="ap-south-1")
sagemaker = boto3.client("sagemaker", region_name="ap-south-1")

bucket_name = "rf-housing-model1"
role_arn = "arn:aws:iam::767261813063:role/SageMakerExecutionRoleForHousePrice"
image_uri = "720646828776.dkr.ecr.ap-south-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3"

model_name = "rf-housing-model-native"
config_name = "rf-serverless-config-native"
endpoint_name = "rf-housing-endpoint"

# 1. Build a lean tarball containing strictly model.joblib
print("Packaging clean model.joblib artifact...")
with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("model.joblib", arcname="model.joblib")

# 2. Upload to S3
print("Uploading clean tarball to S3...")
s3.upload_file("model.tar.gz", bucket_name, "model.tar.gz")

# 3. Clean up existing failed endpoint & config
print("Cleaning up old resources...")
try:
    sagemaker.delete_endpoint(EndpointName=endpoint_name)
    time.sleep(5)
except Exception:
    pass

try:
    sagemaker.delete_endpoint_config(EndpointConfigName=config_name)
except Exception:
    pass

# 4. Create Model using native serving (No custom entrypoint/program variables)
print(f"Creating Model: {model_name}...")
sagemaker.create_model(
    ModelName=model_name,
    PrimaryContainer={
        "Image": image_uri,
        "ModelDataUrl": f"s3://{bucket_name}/model.tar.gz"
    },
    ExecutionRoleArn=role_arn
)

# 5. Create Serverless Endpoint Configuration
print("Creating Serverless Endpoint Configuration...")
sagemaker.create_endpoint_config(
    EndpointConfigName=config_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "ServerlessConfig": {
                "MemorySizeInMB": 2048,
                "MaxConcurrency": 2
            }
        }
    ]
)

# 6. Deploy Serverless Endpoint
print("Deploying Serverless Endpoint...")
sagemaker.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=config_name
)

# 7. Poll Status
while True:
    desc = sagemaker.describe_endpoint(EndpointName=endpoint_name)
    status = desc["EndpointStatus"]
    print("Current Status:", status)
    if status == "InService":
        print("\nEndpoint is now InService and ready for inference!")
        break
    elif status == "Failed":
        print("\nFailed reason:", desc.get("FailureReason"))
        break
    time.sleep(15)

Packaging clean model.joblib artifact...
Uploading clean tarball to S3...
Cleaning up old resources...
Creating Model: rf-housing-model-native...
Creating Serverless Endpoint Configuration...
Deploying Serverless Endpoint...
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Failed

Failed reason: Unable to successfully stand up your model within the allotted 180 second timeout. Please ensure that downloading your model artifacts, starting your model container and passing the ping

In [ ]:
!pip install sagemaker -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.5/52.5 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.7/65.7 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.8/104.8 kB 8.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import sagemaker
from sagemaker.sklearn.model import SKLearnModel
import boto3
import time

role_arn = "arn:aws:iam::767261813063:role/SageMakerExecutionRoleForHousePrice"
bucket_name = "rf-housing-model1"
endpoint_name = "rf-housing-endpoint"

sagemaker_session = sagemaker.Session(boto_session=boto3.Session(region_name="ap-south-1"))

# 1. Clean up old endpoint if present
sm_client = boto3.client("sagemaker", region_name="ap-south-1")
try:
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    time.sleep(5)
except Exception:
    pass

# 2. Define the serverless config using the official SDK
from sagemaker.serverless import ServerlessInferenceConfig

serverless_config = ServerlessInferenceConfig(
    memory_size_in_mb=2048,
    max_concurrency=2
)

# 3. Create SKLearnModel pointing to the latest framework version
model = SKLearnModel(
    model_data=f"s3://{bucket_name}/original_model.tar.gz",
    role=role_arn,
    entry_point="code/inference.py" if os.path.exists("code/inference.py") else "inference.py",
    framework_version="1.2-1",
    sagemaker_session=sagemaker_session
)

print("Deploying endpoint via SageMaker SDK...")
predictor = model.deploy(
    serverless_inference_config=serverless_config,
    endpoint_name=endpoint_name
)

print("\n Endpoint successfully deployed!")

ModuleNotFoundError: No module named 'sagemaker.sklearn'

In [ ]:
import json
import joblib
import numpy as np

# Load your model currently in Colab memory
model = joblib.load("model.joblib")

forest_data = {
    "n_features": model.n_features_in_,
    "n_classes": getattr(model, "n_classes_", 1),
    "estimators": []
}

for est in model.estimators_:
    tree = est.tree_
    forest_data["estimators"].append({
        "children_left": tree.children_left.tolist(),
        "children_right": tree.children_right.tolist(),
        "feature": tree.feature.tolist(),
        "threshold": tree.threshold.tolist(),
        "value": tree.value.tolist()
    })

with open("model.json", "w") as f:
    json.dump(forest_data, f)

print("model.json successfully created! Size:", round(len(json.dumps(forest_data)) / 1024 / 1024, 2), "MB")

model.json successfully created! Size: 77.55 MB


In [ ]:
import os
import tarfile
import boto3
import time

s3 = boto3.client("s3", region_name="ap-south-1")
sagemaker = boto3.client("sagemaker", region_name="ap-south-1")

bucket_name = "rf-housing-model1"
role_arn = "arn:aws:iam::767261813063:role/SageMakerExecutionRoleForHousePrice"
image_uri = "720646828776.dkr.ecr.ap-south-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3"

model_name = "rf-housing-model-json"
config_name = "rf-serverless-config-json"
endpoint_name = "rf-housing-endpoint"

# 1. Create the custom inference engine using pure JSON/NumPy
os.makedirs("code", exist_ok=True)
inference_code = '''import os
import json
import numpy as np

class LightweightForest:
    def __init__(self, data):
        self.estimators = data["estimators"]

    def _predict_tree(self, tree, x):
        node = 0
        while tree["children_left"][node] != -1:
            if x[tree["feature"][node]] <= tree["threshold"][node]:
                node = tree["children_left"][node]
            else:
                node = tree["children_right"][node]
        return tree["value"][node][0][0]

    def predict(self, X):
        X = np.asarray(X)
        if X.ndim == 1:
            X = X.reshape(1, -1)
        preds = []
        for x in X:
            tree_preds = [self._predict_tree(tree, x) for tree in self.estimators]
            preds.append(np.mean(tree_preds))
        return np.array(preds)

def model_fn(model_dir):
    json_path = os.path.join(model_dir, "model.json")
    with open(json_path, "r") as f:
        data = json.load(f)
    print("Lightweight Forest loaded successfully from JSON.")
    return LightweightForest(data)

def input_fn(request_body, request_content_type):
    if request_content_type == "application/json":
        data = json.loads(request_body)
        if isinstance(data, dict) and "inputs" in data:
            return np.array(data["inputs"])
        return np.array(data)
    raise ValueError(f"Unsupported content type: {request_content_type}")

def predict_fn(input_data, model):
    return model.predict(input_data)

def output_fn(prediction, accept):
    if accept == "application/json":
        return json.dumps(prediction.tolist()), accept
    raise ValueError(f"Unsupported accept type: {accept}")
'''

with open("code/inference.py", "w") as f:
    f.write(inference_code)

# 2. Package model.json and inference.py
print("Packaging model.tar.gz...")
with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("model.json", arcname="model.json")
    tar.add("code/inference.py", arcname="code/inference.py")

# 3. Upload to S3
print("Uploading model.tar.gz to S3...")
s3.upload_file("model.tar.gz", bucket_name, "model.tar.gz")

# 4. Clean up stale endpoint/config
print("Cleaning up old resources...")
try:
    sagemaker.delete_endpoint(EndpointName=endpoint_name)
    time.sleep(5)
except Exception:
    pass

try:
    sagemaker.delete_endpoint_config(EndpointConfigName=config_name)
except Exception:
    pass

# 5. Create Model
print(f"Creating Model: {model_name}...")
sagemaker.create_model(
    ModelName=model_name,
    PrimaryContainer={
        "Image": image_uri,
        "ModelDataUrl": f"s3://{bucket_name}/model.tar.gz",
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": f"s3://{bucket_name}/model.tar.gz"
        }
    },
    ExecutionRoleArn=role_arn
)

# 6. Create Serverless Endpoint Configuration
print("Creating Serverless Endpoint Configuration...")
sagemaker.create_endpoint_config(
    EndpointConfigName=config_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "ServerlessConfig": {
                "MemorySizeInMB": 2048,
                "MaxConcurrency": 2
            }
        }
    ]
)

# 7. Deploy Endpoint
print("Deploying Serverless Endpoint...")
sagemaker.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=config_name
)

# 8. Poll Status
while True:
    desc = sagemaker.describe_endpoint(EndpointName=endpoint_name)
    status = desc["EndpointStatus"]
    print("Current Status:", status)
    if status == "InService":
        print("\n Endpoint is now InService and ready for inference!")
        break
    elif status == "Failed":
        print("\n Failed reason:", desc.get("FailureReason"))
        break
    time.sleep(15)

Packaging model.tar.gz...
Uploading model.tar.gz to S3...
Cleaning up old resources...
Creating Model: rf-housing-model-json...
Creating Serverless Endpoint Configuration...
Deploying Serverless Endpoint...
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Failed

 Failed reason: Received server error (0) from model with message "An error occurred while handling request as the model process exited.". See https://ap-south-1.console.aws.amazon.com/cloudwatch/home?region=ap-south-1#logEventViewer:group=/aws/sagemaker/Endpoints/rf-housing-endpoint in account 767261813063 for more information.


In [ ]:
import os
import tarfile
import boto3
import time

s3 = boto3.client("s3", region_name="ap-south-1")
sagemaker = boto3.client("sagemaker", region_name="ap-south-1")

bucket_name = "rf-housing-model1"
role_arn = "arn:aws:iam::767261813063:role/SageMakerExecutionRoleForHousePrice"
image_uri = "720646828776.dkr.ecr.ap-south-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3"

model_name = "rf-housing-model-resolved"
config_name = "rf-serverless-config-resolved"
endpoint_name = "rf-housing-endpoint"

# 1. Ensure inference.py is saved in current working directory
os.makedirs("code", exist_ok=True)
inference_code = '''import os
import json
import numpy as np

class LightweightForest:
    def __init__(self, data):
        self.estimators = data["estimators"]

    def _predict_tree(self, tree, x):
        node = 0
        while tree["children_left"][node] != -1:
            if x[tree["feature"][node]] <= tree["threshold"][node]:
                node = tree["children_left"][node]
            else:
                node = tree["children_right"][node]
        return tree["value"][node][0][0]

    def predict(self, X):
        X = np.asarray(X)
        if X.ndim == 1:
            X = X.reshape(1, -1)
        preds = []
        for x in X:
            tree_preds = [self._predict_tree(tree, x) for tree in self.estimators]
            preds.append(np.mean(tree_preds))
        return np.array(preds)

def model_fn(model_dir):
    json_path = os.path.join(model_dir, "model.json")
    if not os.path.exists(json_path):
        for f in os.listdir(model_dir):
            if f.endswith(".json"):
                json_path = os.path.join(model_dir, f)
                break
    with open(json_path, "r") as f:
        data = json.load(f)
    print("Lightweight Forest loaded successfully.")
    return LightweightForest(data)

def input_fn(request_body, request_content_type):
    if request_content_type == "application/json":
        data = json.loads(request_body)
        if isinstance(data, dict) and "inputs" in data:
            return np.array(data["inputs"])
        return np.array(data)
    raise ValueError(f"Unsupported content type: {request_content_type}")

def predict_fn(input_data, model):
    return model.predict(input_data)

def output_fn(prediction, accept):
    if accept == "application/json":
        return json.dumps(prediction.tolist()), accept
    raise ValueError(f"Unsupported accept type: {accept}")
'''

with open("code/inference.py", "w") as f:
    f.write(inference_code)

with open("inference.py", "w") as f:
    f.write(inference_code)

# 2. Package tarball putting inference.py in every discoverable path
print("Packaging model.tar.gz...")
with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("model.json", arcname="model.json")
    tar.add("inference.py", arcname="inference.py")
    tar.add("code/inference.py", arcname="code/inference.py")

# 3. Upload to S3
print("Uploading to S3...")
s3.upload_file("model.tar.gz", bucket_name, "model.tar.gz")

# 4. Clean up existing endpoint and config
print("Cleaning up old resources...")
try:
    sagemaker.delete_endpoint(EndpointName=endpoint_name)
    time.sleep(5)
except Exception:
    pass

try:
    sagemaker.delete_endpoint_config(EndpointConfigName=config_name)
except Exception:
    pass

# 5. Create Model with exact environment variable pairing
# Setting SAGEMAKER_PROGRAM to "inference.py" and SUBMIT_DIRECTORY to /opt/ml/model
# ensures serving_env.module_name parses as "inference" rather than None.
print(f"Creating Model: {model_name}...")
sagemaker.create_model(
    ModelName=model_name,
    PrimaryContainer={
        "Image": image_uri,
        "ModelDataUrl": f"s3://{bucket_name}/model.tar.gz",
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model",
            "PYTHONPATH": "/opt/ml/model:/opt/ml/model/code"
        }
    },
    ExecutionRoleArn=role_arn
)

# 6. Create Serverless Endpoint Configuration
print("Creating Serverless Endpoint Configuration...")
sagemaker.create_endpoint_config(
    EndpointConfigName=config_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "ServerlessConfig": {
                "MemorySizeInMB": 2048,
                "MaxConcurrency": 2
            }
        }
    ]
)

# 7. Deploy Endpoint
print("Deploying Serverless Endpoint...")
sagemaker.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=config_name
)

# 8. Monitor Status
while True:
    desc = sagemaker.describe_endpoint(EndpointName=endpoint_name)
    status = desc["EndpointStatus"]
    print("Current Status:", status)
    if status == "InService":
        print("\n Endpoint is now InService and ready for predictions!")
        break
    elif status == "Failed":
        print("\n Failed reason:", desc.get("FailureReason"))
        break
    time.sleep(15)

Packaging model.tar.gz...
Uploading to S3...
Cleaning up old resources...
Creating Model: rf-housing-model-resolved...
Creating Serverless Endpoint Configuration...
Deploying Serverless Endpoint...
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: InService

 Endpoint is now InService and ready for predictions!


In [ ]:
import requests

api_url = "https://on6l8u1qcb.execute-api.ap-south-1.amazonaws.com/prod"

payload = {
    "inputs": [
        [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]
    ]
}

response = requests.post(api_url, json=payload)

print("Status Code:", response.status_code)
print("Response:", response.text)

Status Code: 500
Response: {"error": "An error occurred (InvalidParameter) when calling the Publish operation: Invalid parameter: Topic Name"}


In [ ]:
import requests

api_url = "https://on6l8u1qcb.execute-api.ap-south-1.amazonaws.com/prod"

payload = {
    "inputs": [
        [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]
    ]
}

response = requests.post(api_url, json=payload)

print("Status Code:", response.status_code)
print("Response:", response.text)

Status Code: 500
Response: {"error": "An error occurred (InvalidParameter) when calling the Publish operation: Invalid parameter: Topic Name"}


In [ ]:
import boto3

sns = boto3.client("sns", region_name="ap-south-1")

response = sns.list_topics()
print("Your Available SNS Topics:")
for topic in response.get("Topics", []):
    print(topic["TopicArn"])

Your Available SNS Topics:
arn:aws:sns:ap-south-1:767261813063:ModelPredictionResults
arn:aws:sns:ap-south-1:767261813063:house-price-alerts


In [ ]:
import boto3

sns = boto3.client("sns", region_name="ap-south-1")

# Replace with your actual email address
my_email = "your-email@example.com"

# 1. Create the topic
topic_response = sns.create_topic(Name="house-price-alerts")
topic_arn = topic_response["TopicArn"]
print("Created Topic ARN:", topic_arn)

# 2. Subscribe email
sub_response = sns.subscribe(
    TopicArn=topic_arn,
    Protocol="email",
    Endpoint=my_email
)
print("Subscription ARN:", sub_response["SubscriptionArn"])
print("Check your email inbox and click 'Confirm subscription'!")

Created Topic ARN: arn:aws:sns:ap-south-1:767261813063:house-price-alerts
Subscription ARN: pending confirmation
Check your email inbox and click 'Confirm subscription'!


In [ ]:
import requests

api_url = "https://on6l8u1qcb.execute-api.ap-south-1.amazonaws.com/prod"

payload = {
    "inputs": [
        [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]
    ]
}

response = requests.post(api_url, json=payload)
print("Status Code:", response.status_code)
print("Response:", response.text)

Status Code: 200
Response: {"predicted_price": [1.5353503999999998], "notification": "Email notification dispatched successfully"}


In [ ]:
import boto3

sm = boto3.client("sagemaker", region_name="ap-south-1")

sm.delete_endpoint(EndpointName="rf-housing-endpoint")
sm.delete_endpoint_config(EndpointConfigName="rf-serverless-config-resolved")
sm.delete_model(ModelName="rf-housing-model-resolved")

print("SageMaker endpoint resources deleted successfully.")

SageMaker endpoint resources deleted successfully.


to re_deploy


In [ ]:
import boto3
import time

sagemaker = boto3.client("sagemaker", region_name="ap-south-1")

bucket_name = "rf-housing-model1"
role_arn = "arn:aws:iam::767261813063:role/SageMakerExecutionRoleForHousePrice"
image_uri = "720646828776.dkr.ecr.ap-south-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3"

model_name = "rf-housing-model-resolved"
config_name = "rf-serverless-config-resolved"
endpoint_name = "rf-housing-endpoint"

# 1. Re-register the Model
print(f"Recreating Model: {model_name}...")
sagemaker.create_model(
    ModelName=model_name,
    PrimaryContainer={
        "Image": image_uri,
        "ModelDataUrl": f"s3://{bucket_name}/model.tar.gz",
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model",
            "PYTHONPATH": "/opt/ml/model:/opt/ml/model/code"
        }
    },
    ExecutionRoleArn=role_arn
)

# 2. Recreate Serverless Endpoint Configuration
print("Recreating Endpoint Configuration...")
sagemaker.create_endpoint_config(
    EndpointConfigName=config_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "ServerlessConfig": {
                "MemorySizeInMB": 2048,
                "MaxConcurrency": 2
            }
        }
    ]
)

# 3. Deploy Endpoint
print("Deploying Serverless Endpoint...")
sagemaker.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=config_name
)

# 4. Wait for InService status
print("Waiting for endpoint to become active...")
while True:
    desc = sagemaker.describe_endpoint(EndpointName=endpoint_name)
    status = desc["EndpointStatus"]
    print("Current Status:", status)
    if status == "InService":
        print("\n Endpoint is back live and ready for demo!")
        break
    elif status == "Failed":
        print("\n Failed reason:", desc.get("FailureReason"))
        break
    time.sleep(15)

Recreating Model: rf-housing-model-resolved...
Recreating Endpoint Configuration...
Deploying Serverless Endpoint...
Waiting for endpoint to become active...
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: InService

 Endpoint is back live and ready for demo!


In [ ]:
import requests

api_url = "https://on6l8u1qcb.execute-api.ap-south-1.amazonaws.com/prod"

payload = {
    "inputs": [
        [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]
    ]
}

response = requests.post(api_url, json=payload)

if response.status_code == 200:
    data = response.json()

    # Extract prediction value
    raw_pred = data.get("predicted_price", [0])
    price_val = raw_pred[0] if isinstance(raw_pred, list) else raw_pred

    # Format (e.g., assuming price is in $100k or raw dollars)
    estimated_price = price_val * 100000 if price_val < 50 else price_val

    print("=" * 40)
    print("      HOUSE PRICE PREDICTION")
    print("=" * 40)
    print(f"Status           : SUCCESS (HTTP {response.status_code})")
    print(f"Raw Model Value  : {price_val:.4f}")
    print(f"Estimated Price  : ${estimated_price:,.2f}")
    print(f"Notification     : {data.get('notification')}")
    print("=" * 40)
else:
    print(f"Error {response.status_code}:", response.text)

      HOUSE PRICE PREDICTION
Status           : SUCCESS (HTTP 200)
Raw Model Value  : 1.5354
Estimated Price  : $153,535.04
Notification     : Email notification dispatched successfully
